In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [ ]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [ ]:
len(documents)

In [ ]:
from minsearch import Index

index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
index.fit(documents)

In [ ]:
index.search(
        "How does the agentic loop keep calling the model until it stops?",
        num_results=1
    )

In [ ]:
from openai import OpenAI

from rag_helper_homework import RAGBase

openai_client = OpenAI()

assistant = RAGBase(index, openai_client)

In [ ]:
answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

In [ ]:
usage = assistant.get_usage()
print(usage)

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
print(f"Number of chunks: {len(chunks)}")

In [ ]:
from minsearch import Index

indexChunks = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
indexChunks.fit(chunks)

In [ ]:
assistantChunks = RAGBase(indexChunks, openai_client)

In [ ]:
answerChunks = assistantChunks.rag("How does the agentic loop keep calling the model until it stops?")
print(answerChunks)

In [ ]:
usageChunks = assistantChunks.get_usage()
print(usageChunks)

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
def search(query: str) -> dict[str, str]:
    return index.search(
            query,
            num_results=5,
        )

In [ ]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

In [ ]:
INSTRUCTIONS = '''
You're a course teaching assistant. Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
'''

In [ ]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [ ]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)